In [1]:
import whisper
import pyannote
from pyannote.audio import Pipeline
import csv
import ffmpeg
import pydub
from pydub import AudioSegment
import os
import torch
from helper_functions import *
import time
from datetime import datetime


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Comment project_path for local vs PISCES

In [6]:
project_path = 'C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//'

#project_path = 'C://Users//paulbeck//Documents//Audio_Local_tests//'

new_audio_files = project_path + 'audio_files'

#Where my Whisper Transcription goes
transcription_output_folder = project_path + 'transcription_train//'

#Where my pre-transcribed file is taken from
transcription_true = project_path + 'transcription_test//'

transcription_valid = project_path + 'transcription_valid//'
temp_audio_clips_folder = project_path + 'audio_clips//'

rttm_folder  = project_path + 'rttm_files//'

wav_folder =  project_path + 'wav_files//'

transcribed_audio_folder = project_path + 'finished_audio//'

pyan_audio_clips = project_path + "pyan_clips//"

In [7]:
new_audio =  os.listdir(new_audio_files)
print(new_audio)

new_audio = new_audio[0]
print(new_audio)

init_aud_file = os.path.join(new_audio_files, new_audio)

aud = new_audio[:new_audio.find('_')]
print(aud)

['PC1-1025-01_S1_2022.02.28.m4a']
PC1-1025-01_S1_2022.02.28.m4a
PC1-1025-01


In [4]:
# LOCAL TEST ONLY
new_audio =  os.listdir(new_audio_files)
print(new_audio)

#new_audio_files = "C://Users//pisces2//Documents//Audio_Transcription_Deidentification//untranscribed_audio//"
#new_audio = 'hamlet_test.m4a'

#new_audio = 'PC1-1025-01_S1_2022.02.28.m4a'
new_audio = 'hamlet_test.m4a.wav'

init_aud_file = os.path.join(new_audio_files, new_audio)
#init_aud_file = new_audio

print("File is", init_aud_file)
#audio file without extentions:
#aud =  os.path.splitext(new_audio)[0]
aud = 'hamlet_test'

['PC1-1025-01_S1_2022.02.28.m4a']
File is C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_files\hamlet_test.m4a.wav


In [9]:

# checking if it is a file
if os.path.isfile(init_aud_file):
    print("Working on:" , init_aud_file)

#create output file variables:
new_wav_file = aud + ".wav"
print("new_wav_file", new_wav_file)
new_rttm_file = aud + ".rttm"
print("new_rttm", new_rttm_file)
transcript_output = aud + ".csv"
print("Transcript output", transcript_output)

convert_to_wav(init_aud_file, new_wav_file)
print("converted to wav")

audio = AudioSegment.from_file(init_aud_file)
#audio = AudioSegment.from_file(new_wav_file)
audio = audio.set_frame_rate(16000)

start_trim = milliseconds_until_sound(audio)
trimmed = audio[start_trim:]
print(" trimmed complete")
export_path = os.path.join(wav_folder, new_wav_file)

#Segment Audio:
segment_length = 5 * 60 * 1000 

#trimmed.export(export_path, format='wav')

first_five_min = trimmed[: segment_length]
first_five_min.export(wav_folder + "first_five_min.wav", format= "wav")

print("all complete")

Working on: C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_files\PC1-1025-01_S1_2022.02.28.m4a
new_wav_file PC1-1025-01.wav
new_rttm PC1-1025-01.rttm
Transcript output PC1-1025-01.csv
converted to wav
 trimmed complete
all complete


In [ ]:
# Create RTTM Files with collapsed speakers. 
from collections import defaultdict
def read_rttm(file_path):
    ''' Reads RTTM File and returns list of tuples, (start, duration, speaker, line_parts)'''

    with open(file_path, 'r') as f:
      lines = [line.strip() for line in f if line.strip()]

    entries = []
    print("Number of lines to read in RTTM is ", len(lines))
    print("Filename in read_rttm", file_path)

    for line in lines: 
       parts = line.split()

       start_time = float(parts[3])
       duration= float(parts[4])

       speaker = parts[7]

       entries.append((start_time, duration, speaker))

    #Sorts by start time. Should be the case anyway
    return sorted(entries, key=lambda x: x[0])

  


def merge_speaker_segments(entries):
    merged_entries = []

    print("Entries before loop length", len(entries))
    for entry in entries:


        if merged_entries and merged_entries[-1][2] == entry[2]:
         
            prev_start, prev_duration, speaker, prev_parts = merged_entries[-1]

            new_duration = (entry[0] + entry[1]) - prev_start

            prev_parts[4] = str(new_duration)

            merged_entries[-1] = (prev_start, new_duration, speaker, prev_parts)

        else:
            merged_entries.append(entry)

    return merged_entries

def alternate_speakers(merged_entries):

    #Holds all speakers as list 
    speakers = defaultdict(list)
    print(merged_entries)
    print("merged entries length", len(merged_entries))

    #For each merged entry, add the third item [speaker] to speakers list
    for entry in merged_entries:
        speakers[entry[2]].append(entry)

    #Get order of speakers as single speaker switches
    ordered_lines = []

    speaker_keys = list(speakers.keys())

    last_speaker = None

    #For each speaker in SPEAKERS list, check if THE SAME as last speaker. 
    # If so, add to ordered_lines 
    print("Length of speaker_keys", len(speaker_keys))

    print(speakers)
    while any(speakers.values()):

        for speaker in speaker_keys:
           # print("Speaker in Speaker Key", speaker)

            if speakers[speaker] and speaker != last_speaker:

                ordered_lines.append(speakers[speaker].pop(0))

                last_speaker = speaker
                break


    return [" ".join(line[3]) for line in ordered_lines]

#Original Approach. 
def alternate_speakers_time(rttm_file,pre_train=False):

    with open(rttm_file, 'r') as f:
      dlines = [line.strip() for line in f if line.strip()]
    #dlines = open(rttm_file).read().splitlines()
    groups = []
    g = []
    lastend = 0
    rtm_length = 0
    #group rttm lines by speaker
    #print("length dlines", len(dlines))

    #For each line of the rttm_file. 
    for d in dlines:

        start_time, end_time, duration = extract_end_time(d)
        rtm_length += end_time - start_time

        #if new speaker, OR if rtm_len > 30: Used for making sure sub-groups are okay for Whisper
        # May need to just have no limit for pre-train

        if pre_train:
            
            #If there is ANYTHING in g, check if g is on speaker_x, and d is NOT the same speaker.
            if g and (g[0].split()[7] != d.split()[7]):

                #If new speaker for d, add old g to groups (grouped speakers)
                groups.append(g)

                #reset g
                g=[]
                rtm_length = 0
        else:
           # print("should not be here")
            if g and (g[0].split()[7] != d.split()[7]) or rtm_length > 20:
                groups.append(g)
                g=[]
                rtm_length = 0
        
        # Add d to g when same speaker, or after resetting g to start new speaker
        g.append(d)


        end_time = end_time * 1000
        if (lastend > end_time):
            groups.append(g)
            g = []
        else:
            lastend = end_time
        
    if g:
        groups.append(g)
        
    return groups

def combine_groups(groups):

    new_rttm_lines = []

    for g in groups:
        
        start, nend_time, duration = extract_end_time(g[0])
        nstart_time, end, duration = extract_end_time(g[-1])
        tokens = g[0].strip().split()

        new_line = tokens
        print(new_line)
        new_line[3] = start
        new_line[4] = duration

        
        new_rttm_lines.append(new_line)
    
    rttm_string = (" ".join(str(item) for item in line) for line in new_rttm_lines)

    return rttm_string




def write_rttm(file_path, lines):
    with open(file_path, 'w') as f:
        for line in lines:
            f.write(line + '\n')


def clean_rttms(old_rttm_file, new_rttm_file):
    print("Start clean")
    # entries = read_rttm(old_rttm_file)
    # print("Start Merged Entries")
    # merged_entries = merge_speaker_segments(entries)
    # print("start alternate speakers")
    # reordered_lines = alternate_speakers(merged_entries)
    # #reordered_lines = merged_entries
    # print("start write rttm to new folder")


    groups = alternate_speakers_time(old_rttm_file,pre_train=True)
    reordered_lines = combine_groups(groups)
    print(reordered_lines)

    write_rttm(new_rttm_file, reordered_lines)

def process_all_rttms(input_folder, output_folder):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        if filename.endswith(".rttm"):
            input_path = os.path.join(input_folder, filename)
            new_filename = "clean_" + filename
            output_path = os.path.join(output_folder, new_filename)

            clean_rttms(input_path, output_path)


def create_initial_rttm(in_wav_folder, out_rttm_folder):

    if not os.path.exists(out_rttm_folder):
        os.makedirs(out_rttm_folder)

    for filename in os.listdir(in_wav_folder):

        print("create initial rttm, filename is ", filename)

        #new_audio = os.listdir(temp_audio_clips_folder)
        #new_file = temp_audio_clips_folder + new_audio[0]


        pre_pyan = time.time()

        # instantiate the pipeline
        pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1", 
        )


        #pipeline.params['min_duration_on'] = 0.5
        #pipeline.params['min_duration_off'] = 0.3

        #new_file = wav_folder + new_wav_file
        # run the pipeline on an audio file

        device = torch.device("cpu")
        pipeline.to(device)
        
        full_path_filename = os.path.join(in_wav_folder, filename)
        print(f"Processing: {full_path_filename}")

        diarization = pipeline(full_path_filename, num_speakers=2)


        post_pyan = time.time()

        pyan_time = (post_pyan - pre_pyan) / 60
        print("Pyannote time", pyan_time)
        # dump the diarization output to disk using RTTM format
        # make temp file? - dont use specialized naming... 
        file_base = os.path.splitext(filename)[0]
        new_rttm_file = file_base + '.rttm'
        output_rttm = os.path.join(out_rttm_folder, new_rttm_file)
        with open(output_rttm, "w") as rttm:
            diarization.write_rttm(rttm)


# Create RTTM files, loop through list of RTTMS, make Cleaned_rttm Files. 

#Creates initial rttms for all files - will take a while. 
#create_initial_rttm(wav_folder, rttm_folder)

clean_rttm_folder  = project_path + 'clean_rttm_files//'
print("Rttms Complete, Starting clean")
# Runs through all rttm files and cleans them. 
process_all_rttms(rttm_folder, clean_rttm_folder )





Rttms Complete, Starting clean
Start clean
['SPEAKER', 'downsampled_fiv_min', '1', '0.031', '0.608', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '4.469', '0.591', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '4.655', '0.067', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '5.583', '3.831', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '12.012', '0.607', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '16.771', '0.405', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '17.176', '1.114', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '19.133', '0.253', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '19.977', '0.067', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'dow

In [10]:
# Create RTTM files, loop through list of RTTMS, make Cleaned_rttm Files. 

#Creates initial rttms for all files - will take a while. 
create_initial_rttm(wav_folder, rttm_folder)

clean_rttm_folder  = project_path + 'clean_rttm_files//'
print("Rttms Complete, Starting clean")
# Runs through all rttm files and cleans them. 
process_all_rttms(rttm_folder, clean_rttm_folder )

create initial rttm, filename is  downsampled_fiv_min.wav


INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\inspect.py:988: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):


Processing: C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//wav_files//downsampled_fiv_min.wav


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)


Pyannote time 7.176634426911672
create initial rttm, filename is  first_five_min.wav
Processing: C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//wav_files//first_five_min.wav


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)


Pyannote time 7.1968672951062524
create initial rttm, filename is  PC1-1025-01.wav
Processing: C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//wav_files//PC1-1025-01.wav


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)


Pyannote time 73.98685550689697
Rttms Complete, Starting clean
Start clean
['SPEAKER', 'downsampled_fiv_min', '1', '0.031', '0.608', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '4.469', '0.591', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '4.655', '0.067', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '5.583', '3.831', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '12.012', '0.607', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '16.771', '0.405', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '17.176', '1.114', '<NA>', '<NA>', 'SPEAKER_01', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '19.133', '0.253', '<NA>', '<NA>', 'SPEAKER_00', '<NA>', '<NA>']
['SPEAKER', 'downsampled_fiv_min', '1', '19.977', '0.067', '<NA>', '<NA>', 'SPEAKER_01', 

In [ ]:
# from pyannote.audio import Pipeline

# new_audio =  os.listdir(wav_folder)
# print(new_audio)
# print(new_wav_file)

# new_audio = os.listdir(temp_audio_clips_folder)
# new_file = temp_audio_clips_folder + new_audio[0]


# pre_pyan = time.time()

# # instantiate the pipeline
# pipeline = Pipeline.from_pretrained(
#    "pyannote/speaker-diarization-3.1", 
#    )


# #pipeline.params['min_duration_on'] = 0.5
# #pipeline.params['min_duration_off'] = 0.3

# #new_file = wav_folder + new_wav_file
# # run the pipeline on an audio file

# device = torch.device("cpu")
# pipeline.to(device)


# diarization = pipeline(new_file, num_speakers=2)


# post_pyan = time.time()

# pyan_time = (post_pyan - pre_pyan) / 60
# print("Pyannote time", pyan_time)
# # dump the diarization output to disk using RTTM format
# # make temp file? - dont use specialized naming... 

# output_rttm = os.path.join(rttm_folder, new_rttm_file)
# with open(output_rttm, "w") as rttm:
#     diarization.write_rttm(rttm)

# post_rttm = time.time()

# post_rttm_file = (post_rttm - post_pyan) / 60
# print("Post rttm", post_rttm_file)
# print("All time for payn", post_rttm - pre_pyan)
# # https://colab.researchgoogle.com/github/Majdoddin/nlp/blob/main/Pyannote_plays_and_whisper_rhymes_v_2_0.ipynb#scrollTo=AMxtBOk4n8IY
# print("Post dia / pyannote ")

# rttm_file = rttm_folder + new_rttm_file
# dlines = open(rttm_file).read().splitlines()
# groups = []
# g = []
# lastend = 0
# rtm_length = 0
# #group rttm lines by speaker
# for d in dlines:

#     start_time, end_time, duration = extract_end_time(d)
#     rtm_length += end_time - start_time

#     #if new speaker, OR if rtm_len > 30
#     if g and (g[0].split()[7] != d.split()[7]) or rtm_length > 20:
#         groups.append(g)
#         g=[]
#         rtm_length = 0
    
#     g.append(d)


#     end_time = end_time * 1000
#     if (lastend > end_time):
#         groups.append(g)
#         g = []
#     else:
#         lastend = end_time
    
# if g:
#     groups.append(g)




['downsampled_fiv_min.wav', 'first_five_min.wav', 'hamlet_test.wav']
hamlet_test.wav


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std = sequences.std(dim=-1, correction=1)


Pyannote time 7.070169993241628
Post rttm 4.9857298533121747e-05
All time for payn 424.21319103240967
Post dia / pyannote 


HAve RTTM File, audio segment for whisper.

In [ ]:
def ms_min_to_sec(ms):
    seconds = ms // 1000
    minutes = seconds // 60
    remaining_seconds = seconds % 60
    return f"{minutes}:{remaining_seconds}"


def get_audio_file(new_wav_file):
    audio = AudioSegment.from_file(file = new_wav_file, format = "wav") + 5
    return audio

def segment_audio_file(groups,temp_audio_clips_folder):
    gidx = -1
    seconds_count = 0
    groups = list(groups)
    start_stop_list = []

    for g in groups:
        print(len(g))
        print("here is g", g)
        start, nend_time, duration = extract_end_time(g[0])
        nstart_time, end, duration = extract_end_time(g[-1])
        duration = (end - start ) / 1000
        start_time = int(start * 1000)
        end_time = int(end * 1000)
        if (end-start) > 30:

            print("seg time", end-start)
        gidx += 1
        audio_seg = pydub.AudioSegment.empty()

        audio_seg = audio[start_time:end_time]

            #save to folder for temp audio files: to empty after each run. 
        filename =   str(gidx) + ".wav"
        use_name = temp_audio_clips_folder + filename
        print("export to", use_name)
        audio_seg.export(use_name, format ="wav")

        start_stop_list.append([g[0].split()[7], start_time, end_time, duration])

    return start_stop_list, gidx
    
def run_whisper(start_stop_list,gidx,target_output,temp_audio_clips_folder):

    with open(target_output, "w", newline='', encoding = "utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Speaker', 'Start Time', 'Duration', 'Transcription'])
        model = whisper.load_model("large")
        for i in range(gidx + 1):
            #get from audio clips foldery
            audiof = temp_audio_clips_folder + str(i) +  '.wav'
            print("audio clips from ", audiof)
            
            result = model.transcribe(audio = audiof, language = 'en', word_timestamps=True)
            line_items = start_stop_list[i]
            human_time = ms_min_to_sec(line_items[1])
            # if there is any text at all... 
            if result['text'] != "":

                writer.writerow([line_items[0], line_items[1], line_items[3], human_time, result["text"]])
        

    #empty temp folder:
    for temp_audio in os.listdir(temp_audio_clips_folder):
        full_path = os.path.join(temp_audio_clips_folder, temp_audio)
        if os.path.isfile(full_path):
            os.remove(full_path)
        
    print("deleted temp audio")

In [ ]:
target_output = transcription_output_folder + "_v2" + transcript_output

#Get audio, first 5 minutes. 
new_audio =  os.listdir(wav_folder)
print(new_audio)

new_audio = new_audio[0]
print('wav', new_audio)
test_wav = wav_folder + new_audio

#Get audio, first 5 minutes. 
new_audio =  os.listdir(rttm_folder)
print(new_audio)

new_audio = new_audio[0]
print('old_rttm', new_audio)
old_rttm = rttm_folder + new_audio

#Get groups:
groups = alternate_speakers_time(old_rttm)
print(len(groups))
groups = list(groups)
print(groups)
audio = get_audio_file(test_wav)

start_stop, gid = segment_audio_file(groups, temp_audio_clips_folder)
run_whisper(start_stop,gid,target_output,temp_audio_clips_folder)
# print(start_stop, gid)

['downsampled_fiv_min.wav', 'first_five_min.wav', 'PC1-1025-01.wav']
wav downsampled_fiv_min.wav
['downsampled_fiv_min.rttm', 'first_five_min.rttm', 'PC1-1025-01.rttm']
old_rttm downsampled_fiv_min.rttm
50
[['SPEAKER downsampled_fiv_min 1 0.031 0.608 <NA> <NA> SPEAKER_00 <NA> <NA>', 'SPEAKER downsampled_fiv_min 1 1.634 0.405 <NA> <NA> SPEAKER_00 <NA> <NA>', 'SPEAKER downsampled_fiv_min 1 2.410 1.029 <NA> <NA> SPEAKER_00 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 4.469 0.591 <NA> <NA> SPEAKER_01 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 4.655 0.067 <NA> <NA> SPEAKER_00 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 5.583 3.831 <NA> <NA> SPEAKER_01 <NA> <NA>', 'SPEAKER downsampled_fiv_min 1 9.886 6.547 <NA> <NA> SPEAKER_01 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 12.012 0.607 <NA> <NA> SPEAKER_00 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 16.771 0.405 <NA> <NA> SPEAKER_00 <NA> <NA>'], ['SPEAKER downsampled_fiv_min 1 17.176 1.114 <NA> <NA> SPEAKER_01 <NA> <NA>'], ['SPEAKER downsa

c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkp

audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//0.wav


c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//1.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//2.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//3.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//4.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//5.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//6.wav
audi

c:\Users\paulbeck\Documents\GitHub\Audio_Transcription_Deidentification\.venv\Lib\site-packages\whisper\timing.py:210: UserWarning: std_mean(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  std, mean = torch.std_mean(weights, dim=-2, keepdim=True, unbiased=False)


audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//10.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//11.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//12.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//13.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//14.wav
audio clips from  C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//15.wa

: 

In [125]:
#Check if clean_rttm speaker changes match speaker changes for transcription. three WAYS
# 1. No crossover for speaker labels. 
# 2. Switch speaker also matches. 
# 3. does length of each speaker list match?
# Should eventually also do a whisper transcription


#Create two lists, rttm_speakers, transcription_speakers 

#do the rows even aling

# zip list, 

#Get RTTM 

new_audio =  os.listdir(clean_rttm_folder)
print(new_audio)

new_audio = new_audio[0]
print('clean_rttm', new_audio)
test_clean_rttm = clean_rttm_folder + new_audio

entries = read_rttm(test_clean_rttm)

rttm_speakers = [ent[2] for ent in entries ]
print(len(rttm_speakers), "length of speakers rttm")

#Validated Transcription

new_audio =  os.listdir(transcription_valid)
print('transcription', new_audio)

new_audio = new_audio[1]
print(new_audio)
valid_trans = transcription_valid + new_audio

import pandas as pd
valid_trans_df = pd.read_csv(valid_trans)
valid_trans_speakers = valid_trans_df['Speaker'].tolist()
valid_trans_df_cut = valid_trans_df[:30]
print(len(valid_trans_speakers), 'length valid trans speakers')

['clean_downsampled_fiv_min.rttm', 'clean_first_five_min.rttm', 'clean_PC1-1025-01.rttm']
clean_rttm clean_downsampled_fiv_min.rttm
Number of lines to read in RTTM is  47
Filename in read_rttm C://Users//paulbeck//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//clean_rttm_files//clean_downsampled_fiv_min.rttm
47 length of speakers rttm
transcription ['PC1-1025-01_S1_2022.02.28.txt', 'PC1-1025-01_S1_formatted.csv']
PC1-1025-01_S1_formatted.csv
128 length valid trans speakers


In [124]:
#Length RTTM: 47, first 5 min. 365 for full. 
# Length Trans: 128 for full. 
new_audio =  os.listdir(transcription_output_folder)
print('transcription', new_audio)
new_audio = new_audio[0]
print("test", new_audio)
whisper_test = transcription_output_folder + new_audio
whisper_train = pd.read_csv(whisper_test)
print(len(whisper_train))


whisper_train['Transcription'] = whisper_train['Transcription'].replace(r'^\s*$', pd.NA, regex=True)
whisper_clean= whisper_train.dropna(subset=['Transcription'])

whisper_clean['speaker_change'] = (whisper_clean['Speaker'] != whisper_clean['Speaker'].shift()).cumsum()

combined = whisper_clean.groupby('speaker_change').agg({
    'Speaker': 'first',
    'Start Time': 'first',
    'Duration': 'sum',
    'Transcription': ' '.join,
})


print(len(combined))

transcription ['PC1-1025-01.csv']
test PC1-1025-01.csv
38
29


4/18/25: Test against rttm speakers, see if better aligned, whisper creating additional speaker lines 

Do i even need the times to line up? or can i pre-train with just speaker segments, turns, speach? 

Troubleshooting:
1. use large model for making stronger training files. Correct on own.
2. Unsupervised, match transcribed vs transcript, extract times.  

In [131]:
# 47 vs 50 
print(len(whisper_clean), len(valid_trans_df_cut))
df_merged = pd.merge(combined, valid_trans_df_cut, left_index=True, right_index=True, how='outer')

df_rttm_merged = pd.merge()

38 30


In [132]:
print(df_merged)
df_merged.to_csv('test.csv', index=False)

     Speaker_x  Start Time  Duration  \
0          NaN         NaN       NaN   
1   SPEAKER_00        31.0  0.003408   
2   SPEAKER_01      4469.0  0.011441   
3   SPEAKER_00     12012.0  0.000607   
4   SPEAKER_01     17176.0  0.001114   
5   SPEAKER_00     19133.0  0.000845   
6   SPEAKER_01     20973.0  0.000844   
7   SPEAKER_00     21817.0  0.003121   
8   SPEAKER_01     24550.0  0.018664   
9   SPEAKER_00     41898.0  0.000675   
10  SPEAKER_01     44092.0  0.002042   
11  SPEAKER_00     46758.0  0.003426   
12  SPEAKER_01     50622.0  0.001468   
13  SPEAKER_00     52732.0  0.050101   
14  SPEAKER_01    102023.0  0.000827   
15  SPEAKER_00    104285.0  0.004910   
16  SPEAKER_01    109364.0  0.001536   
17  SPEAKER_00    111710.0  0.000371   
18  SPEAKER_01    113532.0  0.018326   
19  SPEAKER_00    132145.0  0.068497   
20  SPEAKER_01    201147.0  0.000422   
21  SPEAKER_00    202868.0  0.022529   
22  SPEAKER_01    225481.0  0.001772   
23  SPEAKER_00    227675.0  0.001519   
